<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [ ]:
'''
REFFRENCE CODE

# Step 2: Create an AsyncOpenAI object for that endpoint
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Step 3: Create a model object to provide when creating an Agent
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
grok_3_model = OpenAIChatCompletionsModel(model="grok-3-mini-beta", openai_client=openrouter_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)
grok_3_via_openrouter_model = OpenAIChatCompletionsModel(model="x-ai/grok-3-mini-beta", openai_client=openrouter_client)
llama_3_2_local_model = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)
'''


In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

import os
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json


In [ ]:
reader = PdfReader("../../twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
with open("../../twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

## LLM to Answer
### Groq

## System Prompt.


In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer to questions that are related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
groq_base_url = os.getenv("GROQ_BASE_URL")
Ollama_base_url = os.getenv("OLLAMA_BASE_URL")
Ollama_api_key = os.getenv("OLLAMA_API_KEY")

In [ ]:
groq_client = OpenAI(
    base_url=groq_base_url,
    api_key=groq_api_key
)


In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

## LLM to Evaluate
### ollama


Ollama


In [ ]:
!ollama pull phi3.5:latest


In [ ]:
# Ollama client (OpenAI-compatible local endpoint)
ollama_client = OpenAI(
    base_url=Ollama_base_url,
    api_key=Ollama_api_key,
)


## Evaluation Prompt

In [ ]:
evaluator_prompt = """
...
You are an evaluator for a professional digital-twin assistant.

PERSON CONTEXT:
{person_context}

USER MESSAGE:
{user_message}

ASSISTANT RESPONSE:
{assistant_response}

Evaluate the response using these rules:
1. It must be grounded in PERSON CONTEXT.
2. It must not invent facts.
3. It must stay related to the person's career, background, skills, or experience.
4. If the user asks something off-topic, the response should redirect to professional topics.
5. If the context does not contain the answer, the assistant should say it does not know.
evaluator_prompt = 


Return ONLY valid JSON in exactly this shape:
{{
  "grounding": 1,
  "relevance": 1,
  "unknown_handling": 1,
  "quality": 1,
  "hallucination": false,
  "pass": true,
  "reason": "short explanation"
}}

...
"""

In [ ]:
def chat(message, history):
    # Gradio history can contain extra fields such as metadata.
    # Groq accepts only the message fields we need here.
    clean_history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = [
        {"role": "system", "content": system_prompt},
        *clean_history,
        {"role": "user", "content": message},
    ]

    # 1) Groq generates a response and may request tools.
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            email = arguments.get("email")

            if email:
                record_email_tool(email)
                tool_result = "Email recorded"
            else:
                tool_result = "No email address was provided"

            messages.append({
                "role": "tool",
                "content": tool_result,
                "tool_call_id": tool_call.id,
            })

        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=tools,
        )

    final_reply = response.choices[0].message.content

    #Ollama / phi3.5 evaluates the final user-facing Groq response.
    evaluation_prompt = evaluator_prompt.format(
        person_context=summary + "\n\n" + linkedin,
        user_message=message,
        assistant_response=final_reply,
    )

    evaluation = ollama_client.chat.completions.create(
        model="phi3.5:latest",
        messages=[{"role": "user", "content": evaluation_prompt}],
    )

    evaluation_text = evaluation.choices[0].message.content or ""

    # phi3.5 should return JSON. If it adds markdown fences, remove them first.
    cleaned_evaluation = evaluation_text.strip()
    if cleaned_evaluation.startswith("```"):
        cleaned_evaluation = cleaned_evaluation.strip("`").strip()
        if cleaned_evaluation.lower().startswith("json"):
            cleaned_evaluation = cleaned_evaluation[4:].strip()

    try:
        evaluation_result = json.loads(cleaned_evaluation)
    except json.JSONDecodeError:
        print("Evaluator returned invalid JSON:")
        print(evaluation_text)
        # Fail open so a formatting mistake in the local evaluator does not break the chat UI.
        return final_reply

    print("Evaluator result:", evaluation_result)

    # Return only responses that pass the evaluator.
    if evaluation_result.get("pass") is True:
        return final_reply

    return (
        "I can only answer questions related to this person's professional "
        "background, career, skills, and experience."
    )


In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>